# Systèmes agentiques composables — Perception → Action

**Notebook** : AgentSystems-01
**Série** : AgentSystems (projet cross-séries — EPIC #26930)
**Durée** : ~1 h 30
**Prérequis** : Python 3.10+, OOP (classes, polymorphisme, duck typing)

## Objectifs d'apprentissage

- Lire la boucle **percevoir → représenter → inférer → décider → vérifier → exécuter → observer** comme un *squelette réutilisable*, pas comme un script.
- Traiter l'**inférence comme un point de composition** : le même agent tourne avec un cerveau déterministe *ou* un LLM, sans que le reste de la boucle change.
- **Réutiliser** les primitives de [`Compression-07b`](../../Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) — la représentation `Segment`, l'invariant de couverture, la boucle de découverte bornée — dans un cadre agentique.
- **Mesurer** l'amortissement du coût : découvrir une politique coûte des inférences, la rejouer en coûte zéro.

> **Position de ce carnet dans le projet.** Il ne ré-invente pas le moteur de
> compression : il en extrait le *squelette d'agent*. Les primitives compression
> sont **réutilisées** (copie pédagogique déclarée — 07b est un notebook, pas un
> module importable) pour montrer la composition ; l'extraction en module
> importable est le pas suivant nommé dans l'EPIC.


In [1]:
import json
import re
from collections import Counter

# Noyau standard uniquement : ce carnet tourne SANS cle API et SANS GPU.
# Le cerveau LLM (Semantic Kernel) est montre plus bas comme point de composition
# interchangeable ; le resultat experimental ci-dessous est produit par le cerveau
# determine, donc reproductible sur n'importe quelle machine.
print("Imports OK : json, re, collections — noyau standard, aucune cle API requise")


Imports OK : json, re, collections — noyau standard, aucune cle API requise


## 1. La boucle perception → action

Un système agentique n'est pas un LLM : c'est un **squelette de traitement** que l'on
**compose** avec un mécanisme d'inférence. La boucle de référence a sept stades :

```mermaid
flowchart LR
    P[Percevoir] --> R[Représenter]
    R --> I[Inférer]
    I --> D[Décider]
    D --> V[Vérifier]
    V -->|couverture OK| E[Exécuter]
    V -->|couverture incomplète| I
    E --> O[Observer]
    O -->|politique amortie| R
```

- **Percevoir / Représenter** : ingérer les entrées brutes dans une *représentation commune* (`Segment`).
- **Inférer / Décider** : proposer un traitement — c'est le **point de composition** (le « cerveau »).
- **Vérifier** : un **invariant** rejette tout plan incomplet (un segment resté sans traitement final).
- **Exécuter / Observer** : appliquer le plan **sans ré-inférer**, puis mesurer le coût.

Le point central : **Inférer** est isolé derrière une *interface*. On y branche un
cerveau déterministe, un LLM, ou les deux — le reste de la boucle ne change pas.
C'est cela, être **composable**.

## 2. Représenter — la perception comme signature de structure

La boucle commence par une **représentation commune**. On réutilise la structure
`Segment` de Compression-07b : un morceau de donnée identifié, portant ses octets,
qualifié par un genre, relié à son origine. La *perception* d'un segment, c'est sa
**signature de structure** — déduite du contenu, pas de l'étiquette.

In [2]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class Segment:
    """Représentation commune d'un morceau de donnée (réutilisée de Compression-07b).

    L'objet que la boucle perçoit et représente : `id` stable, `data` = octets,
    `genre` = qualification, `parent`/`profondeur` = lien vers l'origine.
    """
    id: str
    data: bytes
    genre: str = "brut"
    parent: Optional[str] = None
    profondeur: int = 0


def structure_interne(seg: Segment) -> dict:
    """Signature de structure déduite du CONTENU — la perception du segment.

    C'est ce que le cerveau regarde pour décider. Déduite, pas étiquetée :
    un fichier nommé `.json` qui ne parse pas n'est pas du JSON.
    """
    info = {"octets": len(seg.data), "texte": False, "csv": False,
            "blocs_md": 0, "json": False, "json_obj": False}
    try:
        texte = seg.data.decode("utf-8")
    except UnicodeDecodeError:
        return info
    info["texte"] = True
    t = texte.strip()
    if t[:1] in "{[":
        try:
            info["json_obj"] = isinstance(json.loads(texte), dict)
            info["json"] = True
            return info
        except (json.JSONDecodeError, ValueError):
            pass
    lignes = [l for l in texte.splitlines() if l]
    if lignes and "," in lignes[0]:
        info["csv"] = all(l.count(",") == lignes[0].count(",") for l in lignes)
    info["blocs_md"] = len(re.findall(r"^##? ", texte, re.M))
    return info


def signature(s: dict) -> str:
    """Classe de forme d'un segment — la clé du catalogue amorti.

    Distingue l'objet JSON (decoupable par ses cles) du tableau JSON
    (compressé tel quel) : deux formes, donc deux decisions distinctes.
    """
    if s["json"]:
        return "json_obj" if s["json_obj"] else "json"
    if s["csv"]:
        return "csv"
    if s["blocs_md"]:
        return "markdown"
    return "texte" if s["texte"] else "binaire"


def _corpus_demo() -> dict:
    """Un petit corpus hétérogène, généré déterministiquement : meme entree a
    chaque execution, donc resultat reproductible."""
    ventes = ("nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\n"
              "lait,8,1.10\npain,12,0.80\n").encode("utf-8")
    clients = (json.dumps(
        {"id": 7, "panier": [{"sku": "cafe", "q": 10}, {"sku": "the", "q": 20}]},
        indent=2, ensure_ascii=False) + "\n").encode("utf-8")
    notes = ("# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n"
             "## Risque\nFournisseur unique, lead time 6 semaines.\n").encode("utf-8")
    journal = "".join(f"10:{i:02d}:00 INFO requete {i} ok\n" for i in range(40)).encode("utf-8")
    binaire = bytes(range(256)) * 4
    return {
        "ventes.csv#0": Segment("ventes.csv#0", ventes, "donnees/csv"),
        "clients.json#0": Segment("clients.json#0", clients, "donnees/json"),
        "notes.md#0": Segment("notes.md#0", notes, "texte/markdown"),
        "journal.log#0": Segment("journal.log#0", journal, "texte/log"),
        "brut.bin#0": Segment("brut.bin#0", binaire, "binaire"),
    }


# Démonstration de la perception : pour chaque segment du corpus, on lit sa forme
# interne déduite du contenu (structure_interne) puis sa classe de forme
# (signature) — sans aucune étiquette, seulement à partir des octets.
SEGMENTS = _corpus_demo()
for sid, seg in SEGMENTS.items():
    s = structure_interne(seg)
    print(f"  {sid:16s} {len(seg.data):5d} o   forme={signature(s):9s} "
          f"json={s['json']} csv={s['csv']} blocs_md={s['blocs_md']} texte={s['texte']}")
print(f"\ncorpus percu : {len(SEGMENTS)} segments heterogenes, {len(set(signature(structure_interne(x)) for x in SEGMENTS.values()))} formes distinctes")

  ventes.csv#0        81 o   forme=csv       json=False csv=True blocs_md=0 texte=True
  clients.json#0     127 o   forme=json_obj  json=True csv=False blocs_md=0 texte=True
  notes.md#0          95 o   forme=markdown  json=False csv=False blocs_md=3 texte=True
  journal.log#0     1110 o   forme=texte     json=False csv=False blocs_md=0 texte=True
  brut.bin#0        1024 o   forme=binaire   json=False csv=False blocs_md=0 texte=False

corpus percu : 5 segments heterogenes, 5 formes distinctes


### Lecture du résultat

Cinq segments, **cinq formes distinctes** : le CSV et le JSON sont des structures
*exploitables* (on peut les découper en parties de nature différente), le markdown
est structuré par ses en-têtes, le journal est un texte plat, le binaire n'a pas de
structure lisible. C'est exactement la hétérogénéité que la boucle doit gérer — et
c'est ce qui rendra les décisions **non uniformes** plus loin (le tri n'est pas
trivial : chaque forme appelle une opération différente).

## 3. Inférer — le point de composition

Le stade **Inférer** est isolé derrière une *interface* : `Cerveau.inferer(seg) ->
decision`. C'est ce contrat unique — et rien d'autre — qui rend le cerveau
**interchangeable**. La boucle ne sait pas *qui* infère, seulement *que* quelqu'un
infère. On commence par le cerveau **déterministe**, qui dérive la décision de la
signature perçue : c'est lui qui produira le résultat expérimental.

In [3]:
class Cerveau:
    """Point de composition : le contrat d'un mecanisme d'inference.

    La boucle agentique n'appelle JAMAIS un cerveau en particulier, seulement
    cette interface. C'est ce contrat unique qui rend le cerveau interchangeable
    (deterministe, LLM, hybride). Chaque consultation compte comme une inference.
    """
    def __init__(self):
        self.appels = 0

    def inferer(self, seg: Segment) -> dict:
        raise TypeError("Cerveau est abstrait : specialiser inferer()")


class CerveauDeterministe(Cerveau):
    """Le cerveau par defaut : derive la decision de la structure perçue.

    Joue le role que le LLM joue dans Compression-07b, en deterministe : meme
    perception -> meme decision, donc cout reproductible. La decision est une
    *regle de forme* : les formes exploitables (csv/json/markdown) sont découpées
    pour reduire, le reste est compressé.
    """
    SEUIL_DECOUPE = 64  # un segment trop petit n'a pas d'interet a etre decoupe

    def inferer(self, seg: Segment) -> dict:
        self.appels += 1
        s = structure_interne(seg)
        if len(seg.data) < self.SEUIL_DECOUPE:
            return {"op": "compresser", "codec": "zlib" if s["texte"] else "lzma"}
        forme = signature(s)
        if forme == "json_obj":
            return {"op": "decouper", "mode": "cles_json"}
        if forme == "csv":
            return {"op": "decouper", "mode": "colonnes_csv"}
        if forme == "markdown":
            return {"op": "decouper", "mode": "blocs_markdown"}
        return {"op": "compresser", "codec": "zlib" if s["texte"] else "lzma"}


apercu = CerveauDeterministe()
print(f"cerveau determine par : {type(apercu).__name__}  (seuil decoupe = {apercu.SEUIL_DECOUPE} o)\n")
print("decision derivee de la forme, segment par segment :")
for sid, seg in SEGMENTS.items():
    d = apercu.inferer(seg)
    print(f"  {sid:16s} -> {d['op']:10s} {d.get('mode') or d.get('codec')}")
print(f"\ninférences effectuees pour cet apercu : {apercu.appels}")


cerveau determine par : CerveauDeterministe  (seuil decoupe = 64 o)

decision derivee de la forme, segment par segment :
  ventes.csv#0     -> decouper   colonnes_csv
  clients.json#0   -> decouper   cles_json
  notes.md#0       -> decouper   blocs_markdown
  journal.log#0    -> compresser zlib
  brut.bin#0       -> compresser lzma

inférences effectuees pour cet apercu : 5


### Lecture du résultat

Le cerveau **n'est pas un classifieur en dur** : il *dérive* chaque décision de la
signature perçue. Les formes exploitables (`csv`, `json`, `markdown`) au-delà du
seuil sont **découpées** (l'opération qui *réduit*), le reste est **compressé**
(avec un codec choisi par la nature, texte/binaire). Cinq segments, cinq décisions —
la non-uniformité est là. Et chaque décision a coûté **une inférence** : c'est le
compteur que la boucle va amortir.

## 4. Vérifier — l'invariant de couverture

Le stade **Vérifier** est le garde-fou de la composition : un plan n'est **pas
exécutable** tant qu'il laisse un segment d'origine *sans traitement final*. On
réutilise l'invariant de couverture de Compression-07b (`CouvertureIncomplete` +
`verifier_couverture`) : un segment est couvert s'il est compressé, **ou** s'il est
découpé en enfants *tous* couverts. C'est le **témoin négatif** de l'organe — il
nomme précisément ce qui manque.

In [4]:
class CouvertureIncomplete(RuntimeError):
    """Leve quand un segment original n'a reçu aucun traitement final."""


def _couverture(sid: str, recette: dict, vus: set) -> tuple:
    """Un segment est couvert s'il est compressé, ou découpé en enfants tous couverts."""
    if sid in vus:
        raise CouvertureIncomplete(f"cycle dans la recette autour de {sid}")
    vus = vus | {sid}
    entree = recette.get(sid)
    if entree is None:
        return False, [sid]
    if entree["op"] == "compresser":
        return True, []
    manquants = []
    for enfant in entree["enfants"]:
        ok, absents = _couverture(enfant, recette, vus)
        if not ok:
            manquants.extend(absents)
    return not manquants, manquants


def verifier_couverture(racines: list, recette: dict) -> None:
    """Invariant : leve en nommant tout segment original non traité."""
    manquants = []
    for sid in racines:
        ok, absents = _couverture(sid, recette, set())
        if not ok:
            manquants.extend(absents)
    if manquants:
        raise CouvertureIncomplete(
            f"{len(manquants)} segment(s) sans traitement final : {manquants[:6]}")


# Les operations inversibles (réutilisées de 07b) : decouper / recoller, et codecs.
import zlib
import lzma

CODECS = {
    "zlib": (lambda d: zlib.compress(d, 9), zlib.decompress),
    "lzma": (lambda d: lzma.compress(d, preset=9), lzma.decompress),
    "store": (lambda d: d, lambda d: d),
}


def _decouper(seg: Segment, mode: str) -> list:
    """Operation inversible : un segment macro -> des fragments qualifies."""
    texte = seg.data.decode("utf-8")  # les modes ci-dessous ne s'appliquent qu'au texte
    if mode == "blocs_markdown":
        bornes = [0, *[m.start() for m in re.finditer(r"^##? ", texte, re.M)], len(texte)]
        morceaux = [texte[a:b] for a, b in zip(bornes, bornes[1:])]
        kids = [Segment(f"{seg.id}.b{i}", m.encode("utf-8"), f"{seg.genre}/bloc",
                        seg.id, seg.profondeur + 1)
                for i, m in enumerate(morceaux) if m.strip()]
    elif mode == "colonnes_csv":
        lignes = [l for l in texte.splitlines() if l]
        cols = lignes[0].split(",")
        mat = [l.split(",") for l in lignes]
        kids = [Segment(f"{seg.id}.c{j}",
                         "\n".join(r[j] for r in mat).encode("utf-8"),
                         f"{seg.genre}/col:{cols[j]}", seg.id, seg.profondeur + 1)
                for j in range(len(cols))]
    elif mode == "cles_json":
        obj = json.loads(texte)
        kids = [Segment(f"{seg.id}.k{i}",
                         json.dumps(v, indent=2, ensure_ascii=False).encode("utf-8"),
                         f"{seg.genre}/cle:{k}", seg.id, seg.profondeur + 1)
                for i, (k, v) in enumerate(obj.items())]
    else:
        raise ValueError(f"mode de decoupe inconnu : {mode}")
    return kids


def _recoller(mode: str, enfants: list) -> bytes:
    """L'inverse d'un decoupage : reconstruit les octets du segment parent."""
    if mode == "blocs_markdown":
        return b"".join(e.data for e in enfants)
    if mode == "colonnes_csv":
        mat = [e.data.decode("utf-8").split("\n") for e in enfants]
        n = len(mat[0])
        lignes = [",".join(mat[c][row] for c in range(len(mat))) for row in range(n)]
        return ("\n".join(lignes) + "\n").encode("utf-8")
    if mode == "cles_json":
        obj = {e.genre.split(":")[-1]: json.loads(e.data.decode("utf-8")) for e in enfants}
        return (json.dumps(obj, indent=2, ensure_ascii=False) + "\n").encode("utf-8")
    raise ValueError(f"mode de recolle inconnu : {mode}")


# Contrat d'inversibilite (temoin de l'organe) : decouper puis recoller redonne
# les octets d'origine, pour chaque mode present dans le corpus.
for seg in SEGMENTS.values():
    for mode in ("blocs_markdown", "colonnes_csv", "cles_json"):
        try:
            if mode == "colonnes_csv" and not structure_interne(seg)["csv"]:
                continue
            if mode == "cles_json" and not structure_interne(seg)["json"]:
                continue
            if mode == "blocs_markdown" and not structure_interne(seg)["blocs_md"]:
                continue
            enfants = _decouper(seg, mode)
            assert enfants, f"decoupe vide : {mode}"
            assert _recoller(mode, enfants) == seg.data, f"non inversible : {mode}"
        except (UnicodeDecodeError, ValueError, json.JSONDecodeError):
            continue  # mode inapplicable a ce segment : on passe au suivant
print(f"codecs : {sorted(CODECS)} ; decoupes/recolles verifiés inversibles sur le corpus")
for nom, (c, d) in CODECS.items():
    assert d(c(b"temoin de codec " * 5)) == b"temoin de codec " * 5, f"codec non inversible : {nom}"
print("codecs inversibles : OK")


codecs : ['lzma', 'store', 'zlib'] ; decoupes/recolles verifiés inversibles sur le corpus


codecs inversibles : OK


### Lecture du résultat

Deux familles d'opérations, toutes **inversibles** (le témoin négatif passe) : les
*découpes* (csv/json/markdown) et les *codecs* (zlib/lzma/store). L'invariant de
couverture qui vient d'être défini est le **garde-fou de la composition** : il sera
déclenché plus bas sur un plan *délibérément* incomplet, pour montrer que la boucle
**rejette** un plan qu'elle ne peut pas exéuter en toute sécurité.

## 5. Le squelette — `percevoir → … → observer`

On assemble maintenant les stades en un **squelette** `AgentTriage`. Il compose :
la représentation (`Segment`), un cerveau (le point de composition), l'invariant de
couverture (la vérification), et l'application de la recette (l'exécution). Le
résultat clé est dans `decouvrir` / `rejouer` : la découverte **paie** le
catalogue `forme → décision` en inférences ; la rejeu le **consomme** sans
ré-inférer.

In [5]:
class AgentTriage:
    """Le squelette agentique : percevoir -> representer -> inferer -> decider
    -> verifier -> executer -> observer.

    Compose la representation (Segment), un cerveau (point de composition),
    l'invariant de couverture (verification) et l'application de la recette
    (execution). L'inference est isolee derriere `cerveau`.
    """
    def __init__(self, cerveau: Cerveau, borne: int = 64):
        self.cerveau = cerveau
        self.borne = borne
        self.catalogue: dict = {}   # forme -> decision  (le savoir amorti)
        self.recette: dict = {}     # id segment -> decision
        self.appel_decouverte = 0

    # percevoir + representer : l'entree est deja des Segment (perception faite)
    def decouvrir(self, segments: list) -> dict:
        """Apprend le catalogue forme->decision en parcourant le corpus.

        Chaque consultation du cerveau compte. Si le parcours atteint la borne
        sans epuiser la file, la decouverte ECHOUE (safety bound) : on ne laisse
        jamais l'agent tourner a l'infini.
        """
        self.catalogue, self.recette = {}, {}
        self.appel_decouverte = 0
        file = list(segments)
        while file:
            if self.appel_decouverte >= self.borne:
                raise RuntimeError(
                    f"decouverte interrompue a {self.appel_decouverte} inférences, "
                    f"{len(file)} segment(s) en attente — safety bound.")
            seg = file.pop(0)
            dec = self.cerveau.inferer(seg)          # inferer + decider (compté)
            self.appel_decouverte += 1
            self.catalogue[signature(structure_interne(seg))] = dec
            if dec["op"] == "decouper":
                enfants = _decouper(seg, dec["mode"])
                # la recette porte les ENFANTS reels, pas seulement la decision :
                # c'est cette liste que l'invariant de couverture parcourt.
                self.recette[seg.id] = {"op": "decouper", "mode": dec["mode"],
                                         "enfants": [e.id for e in enfants]}
                file.extend(enfants)
            else:
                self.recette[seg.id] = dec
        verifier_couverture([s.id for s in segments], self.recette)  # verifier
        return self.recette

    def rejouer(self, segments: list) -> dict:
        """Applique le catalogue appris a de nouveaux segments, SANS re-inferer.

        C'est l'amortissement : la decouverte a paye le catalogue, la rejeu le
        consomme a cout d'inference NUL.
        """
        return {seg.id: self.catalogue[signature(structure_interne(seg))]
                for seg in segments}


### Lecture du résultat

Le squelette tient en une classe. Notez la place exacte de chaque stade : `decouvrir`
fait **percevoir→représenter→inférer→décider→vérifier** d'un bloc (la vérification
est *dans* la découverte — un plan incomplet n'est jamais accepté), et `rejouer`
fait **exécuter** le savoir acquis **sans passer par l'inférence**. `observer`,
c'est le compteur `appel_decouverte` qu'on va lire dans la section suivante.

## 6. Exécuter + Observer — l'amortissement du coût

On laisse le squelette **découvrir** la politique sur le corpus, puis on le fait
**rejouer** sur un *deuxième* corpus de même forme. L'observation est le contraste
de coût entre les deux phases.

In [6]:
# Découverte : on instancie l'agent (cerveau déterministe, borne 64 octets), on
# lui fait percevoir le corpus, puis on lit le catalogue amorti de formes et la
# recette — ce que l'agent a déduit de ce qu'il sait faire pour chaque forme.
agent = AgentTriage(CerveauDeterministe(), borne=64)
agent.decouvrir(list(SEGMENTS.values()))
print(f"DECOUVERTE (1er corpus, {len(SEGMENTS)} racines) :")
print(f"   inférences effectuees : {agent.appel_decouverte}")
print(f"   catalogue amorti ({len(agent.catalogue)} formes) :")
for forme, dec in agent.catalogue.items():
    print(f"      {forme:9s} -> {dec['op']:10s} {dec.get('mode') or dec.get('codec')}")
print(f"   recette (segments couverts) : {len(agent.recette)} entrees")

DECOUVERTE (1er corpus, 5 racines) :
   inférences effectuees : 13
   catalogue amorti (6 formes) :
      csv       -> decouper   colonnes_csv
      json_obj  -> decouper   cles_json
      markdown  -> compresser zlib
      texte     -> compresser zlib
      binaire   -> compresser lzma
      json      -> compresser zlib
   recette (segments couverts) : 13 entrees


In [7]:
# Un DEUXIEME corpus, meme forme, donnees differentes : la rejeu consomme le
# catalogue appris, sans re-inferer une seule fois.
ventes2 = ("nom,quantite,prix\ncafe,4,2.50\nbiere,16,3.00\n").encode("utf-8")
nouveau = {"ventes2.csv#0": Segment("ventes2.csv#0", ventes2, "donnees/csv")}
plan = agent.rejouer(list(nouveau.values()))
print(f"REJEU (2e corpus, {len(nouveau)} racine) :")
for sid, dec in plan.items():
    print(f"   {sid:16s} -> {dec['op']:10s} {dec.get('mode') or dec.get('codec')}   (0 inférence)")
print(f"   inférences effectuees : 0  (le catalogue a ete consulte, pas le cerveau)")


REJEU (2e corpus, 1 racine) :
   ventes2.csv#0    -> decouper   colonnes_csv   (0 inférence)
   inférences effectuees : 0  (le catalogue a ete consulte, pas le cerveau)


In [8]:
# L'observation : le cout d'inference est PAYE une fois (decouverte) puis amorti
# a zero (rejeu). C'est la these de Compression-07b, reformulee comme propriete
# d'un agent composable : la connaissance forme->decision est un ACTIF amortissable.
print("OBSERVATION — amortissement du cout d'inference :")
print(f"   decouverte : {agent.appel_decouverte} inférences   (politique apprise)")
print(f"   rejeu      : 0 inférences       (politique consommee)")
print(f"   -> sur 1000 corpus de meme forme, on re-infererait {agent.appel_decouverte} fois, pas 1000 x {agent.appel_decouverte}.")


OBSERVATION — amortissement du cout d'inference :
   decouverte : 13 inférences   (politique apprise)
   rejeu      : 0 inférences       (politique consommee)
   -> sur 1000 corpus de meme forme, on re-infererait 13 fois, pas 1000 x 13.


### Lecture du résultat

C'est le **résultat expérimental** du carnet : la découverte a coûté
`{agent.appel_decouverte}` inférences pour apprendre le catalogue
`forme → décision` ; la rejeu sur un corpus *nouveau* de même forme a coûté **zéro**
inférence, car elle a consulté le catalogue, pas le cerveau. L'amortissement est
réel et mesurable : la connaissance est un actif qui se paie une fois. C'est ce que
« composition » veut dire concrètement — le coût d'inférence est *déplacé* de
l'exécution vers la découverte, et l'exécution devient gratuite en inférences.

## 7. La vérification en action — rejeter un plan incomplet

Le garde-fou n'est pas théorique : on le **déclenche**. On construit un plan où un
segment est découpé mais dont *un enfant* est laissé sans traitement final, et on
le passe à `verifier_couverture`. La boucle **doit** le rejeter en nommant
précisément ce qui manque — c'est la capacité de *vérification* d'un système
composable, qui empêche d'exécuter un plan à moitié formé.

In [9]:
# PLAN INCOMPLET : x.md est decoupe en deux blocs, mais b1 n'a AUCUN traitement
# final. L'invariant de couverture doit rejeter le plan en nommant b1.
plan_casse = {
    "x.md#0":     {"op": "decouper", "mode": "blocs_markdown",
                   "enfants": ["x.md#0.b0", "x.md#0.b1"]},
    "x.md#0.b0":  {"op": "compresser", "codec": "zlib"},
    # x.md#0.b1 : absent de la recette -> non couvert
}
try:
    verifier_couverture(["x.md#0"], plan_casse)
    print("BUG : le plan incomplet n'a PAS ete rejete")
except CouvertureIncomplete as e:
    print(f"VÉRIFICATION : le plan incomplet est REJETE -> {e}")

# CONTROLE POSITIF : le plan decouvert, lui, est complet -> l'invariant le laisse passer.
verifier_couverture([s.id for s in SEGMENTS.values()], agent.recette)
print("VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer")


VÉRIFICATION : le plan incomplet est REJETE -> 1 segment(s) sans traitement final : ['x.md#0.b1']
VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer


### Lecture du résultat

Le même invariant **rejette** le plan à moitié formé (il nomme `x.md#0.b1`, l'enfant
orphelin) et **laisse passer** le plan découvert (tous les segments y ont un
traitement final). C'est la différence entre *avoir un plan* et *avoir un plan
exécutable* — et c'est ce qui autorise la composition : on ne branche un
mécanisme d'inférence que derrière une porte de vérification qui garantit que le
plan produit est complet.

## 8. La safety bound — arrêter une découverte divergente

La deuxième capacité de sécurité d'un système composable : **ne jamais tourner à
l'infini**. On branche un cerveau *pathologique* qui découpe toujours, sans
produire de traitement final — le parcours ne s'épuise jamais. Le squelette doit
l'**arrêter** à la borne d'appels, au lieu de laisser l'agent s'épuiser.

In [10]:
class CerveauDivergent(Cerveau):
    """Cerveau pathologique : decoupe TOUJOURS, ne produit jamais de feu final.

    Le parcours ne s'epuise jamais : c'est precisement le cas que la safety bound
    doit arreter, au lieu de laisser l'agent tourner a l'infini.
    """
    def inferer(self, seg: Segment) -> dict:
        self.appels += 1
        return {"op": "decouper", "mode": "blocs_markdown"}


divergent = AgentTriage(CerveauDivergent(), borne=8)
try:
    # texte SANS en-tete : blocs_markdown ne produit qu'un seul bloc = le segment
    # lui-meme, re-decoupe a l'identique -> le parcours ne s'epuise jamais.
    divergent.decouvrir([Segment("boucle.txt#0",
                                  b"texte sans en-tete, a decouper a l'infini\n", "texte")])
    print("BUG : la divergence n'a PAS ete arretee")
except RuntimeError as e:
    print(f"SAFETY BOUND : la decouverte divergente est arretee -> {e}")


SAFETY BOUND : la decouverte divergente est arretee -> decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound.


### Lecture du résultat

Le cerveau divergent découpe indéfiniment (un bloc sans en-tête se « découpe » en
un seul bloc identique) : sans borne, l'agent tournerait pour toujours. Le
squelette l'**arrête à 8 inférences** et signale l'échec de la découverte. C'est
la contrepartie de la composition : on accepte de brancher n'importe quel
mécanisme d'inférence *parce que* le squelette porte les deux garde-fous — la
**vérification** de complétude (section 7) et la **safety bound** (ici). Sans
eux, « composable » ne serait que « instable ».

## 9. Le second cerveau — le LLM, point de composition

On montre maintenant concrètement le **point de composition** : le *même*
`AgentTriage` accepte un cerveau **LLM** (Semantic Kernel, tel que
Compression-07b l'emploie) aussi bien que le cerveau déterministe, parce qu'il ne
voit que l'interface `Cerveau`. Le cerveau LLM est **déclaré, pas invoqué** ici —
le carnet reste reproductible sans clé API — et c'est exactement le propos : le
cerveau est une *pièce* amovible, pas le squelette.

In [11]:
class CerveauLLM(Cerveau):
    """Point de composition : un LLM derriere la MEME interface Cerveau.

    En production, `inferer` construirait le kernel Semantic Kernel (un kernel
    neuf par decision, cf Compression-07b) et lirait l'appel de fonction
    compresser/decouper. Ici il est DECLARE, pas invoque : le carnet reste
    reproductible sans cle, et c'est precisement le point illustre — le cerveau
    est une piece, pas le squelette.
    """
    def inferer(self, seg: Segment) -> dict:
        raise TypeError("CerveauLLM.inferer n'est pas cablé dans ce carnet (pas de cle API)")


# Le point de composition, concret : le meme squelette accepte les deux cerveaux,
# car il ne teste que l'interface Cerveau (duck typing).
for nom, cer in [("deterministe", CerveauDeterministe()), ("LLM", CerveauLLM())]:
    print(f"   AgentTriage accepte un cerveau {nom:14s} : {isinstance(cer, Cerveau)}")
print("\nla boucle est agnostique du mecanisme d'inference : c'est la composition.")
print("le resultat experimental ci-dessus a ete produit par le cerveau determine ;")
print("brancher CerveauLLM changerait le COAT, pas le squelette ni les garde-fous.")


   AgentTriage accepte un cerveau deterministe   : True
   AgentTriage accepte un cerveau LLM            : True

la boucle est agnostique du mecanisme d'inference : c'est la composition.
le resultat experimental ci-dessus a ete produit par le cerveau determine ;
brancher CerveauLLM changerait le COAT, pas le squelette ni les garde-fous.


### Lecture du résultat

Le même squelette accepte les deux cerveaux — le test `isinstance(cer, Cerveau)`
passe pour les deux, et c'est tout ce que la boucle exige. La composition est
réelle : **brancher un LLM change le coût d'inférence, pas le squelette, pas la
vérification, pas la safety bound.** Le résultat expérimental de ce carnet a été
produit par le cerveau déterministe (reproductible, sans clé) ; en production on
y brancherait `CerveauLLM`, et les sections 6 à 8 garderaient leur sens à
l'identique.

## 10. Exercices

Trois exercices, un par capacité du squelette : étendre la **perception**, ajouter
une **opération inversible**, et exploiter le **point de composition**.

### Exercice 1 — étendre la perception

Le corpus ne détecte que le CSV (virgules). Ajoutez la détection du **TSV**
(colonnes séparées par des tabulateurs) dans `structure_interne`, puis une
signature `"tsv"` dans `signature()` (avant `"csv"`).

- `# Etape 1` : dans `structure_interne`, poser `info["tsv"]` à `True` quand chaque ligne non vide partage le même nombre de `\t` que la première.
- `# Etape 2` : dans `signature`, renvoyer `"tsv"` avant le test `csv`.
- `# Indice` : un TSV a des `\t`, un CSV a des `','` — même logique, autre séparateur.


In [12]:
def structure_tsv(seg: Segment) -> dict:
    # TODO etudiant : renvoyer une signature de structure qui detecte le TSV
    # (colonnes separees par des tabulateurs), comme structure_interne le fait
    # pour le CSV. Doit renvoyer un dict avec au moins la cles "tsv" (bool).
    return None  # TODO etudiant


if structure_tsv(SEGMENTS["ventes.csv#0"]) is None:
    print("Exercice 1 a completer : detection du TSV dans la perception")


Exercice 1 a completer : detection du TSV dans la perception


### Exercice 2 — une opération inversible

Écrivez un codec **RLE** (run-length : `(octet, compte)`) et enregistrez-le dans
`CODECS`. Le contrat d'inversibilité doit tenir : `decompresser(compresser(d)) == d`.

- `# Etape 1` : `compresser_rle` parcourt `donnees` et émet des paires `(octet, compte)` pour chaque run consécutive.
- `# Etape 2` : `decompresser_rle` lit les paires et répète chaque octet `compte` fois.
- `# Indice` : `"aaabbb"` → `[(a,3),(b,3)]` ; attention au run de longueur 1.


In [13]:
def compresser_rle(donnees: bytes):
    # TODO etudiant : encoder les runs consecutives en paires (octet, compte)
    return None  # TODO etudiant


def decompresser_rle(donnees: bytes):
    # TODO etudiant : reconstruire les octets d'origine depuis les paires
    return None  # TODO etudiant


if compresser_rle(b"aaab") is None:
    print("Exercice 2 a completer : codec RLE inversible")


Exercice 2 a completer : codec RLE inversible


### Exercice 3 — le point de composition

Écrivez un `CerveauHybride` qui **route** : les petits segments (moins d'un seuil
d'octets) sont confiés au cerveau déterministe, les gros renvoient
`{"op": "escalade"}` (à traiter par un LLM en production). Comptez les appels
comme les autres cerveaux.

- `# Etape 1` : dans `inferer`, si `len(seg.data) < self.seuil`, déléguer à `self.petit.inferer(seg)`.
- `# Etape 2` : sinon renvoyer `{"op": "escalade"}` et compter l'appel.
- `# Indice` : le hybride *contient* un cerveau — c'est une composition de compositions.


In [14]:
class CerveauHybride(Cerveau):
    def __init__(self, petit: Cerveau, seuil: int = 128):
        super().__init__()
        self.petit = petit
        self.seuil = seuil

    def inferer(self, seg: Segment) -> dict:
        # TODO etudiant : router selon len(seg.data) vs self.seuil
        return None  # TODO etudiant


# Le carnet doit rester executable : ce squelette renvoie None tant qu'incomplet.
print("Exercice 3 a completer : cerveau hybride (routing petit/gros)")


Exercice 3 a completer : cerveau hybride (routing petit/gros)


## 11. À retenir

- **Un agent composable = un squelette + des pièces.** La boucle
  percevoir→représenter→inférer→décider→vérifier→exécuter→observer est le
  squelette ; l'inférence est une *pièce* derrière une interface.
- **L'inférence est un point de composition.** Déterministe, LLM, hybride : le
  squelette, la vérification et la safety bound ne changent pas — seul le **coût**
  d'inférence change.
- **La connaissance s'amortit.** Découvrir une politique `forme → décision` coûte des
  inférences une fois ; la rejouer en coûte zéro. La connaissance est un actif.
- **Composer exige des garde-fous.** Un mécanisme d'inférence ne se branche que
  derrière une **vérification de complétude** (l'invariant de couverture rejette un
  plan à moitié formé) et une **safety bound** (on n'arrête pas une découverte
  divergente).
- **Les primitives se réutilisent.** La représentation, l'invariant et la boucle
  bornée viennent de Compression-07b ; ce carnet les *compose*, il ne les réinvente
  pas.

## Vérifiez votre compréhension

1. Pourquoi le stade **Vérifier** est-il placé *dans* la découverte, et non après
   l'exécution ? *(Réponse repliée ci-dessous.)*
2. Que mesure exactement le compteur `appel_decouverte`, et pourquoi est-il de 0
   pendant `rejouer` ?
3. Quel est le risque précis que la **safety bound** empêche, et qu'arriverait-il
   sans elle sur un cerveau divergent ?

<details><summary>Réponse 1</summary>
Parce qu'un plan incomplet ne doit **jamais** être exécuté : la vérification est
la *porte d'admission* du plan. Si elle était après l'exécution, on aurait déjà
transformé des données avec un plan à moitié formé — trop tard pour rejeter.
</details>

<details><summary>Réponse 2</summary>
Il compte les **consultations du cerveau** (les véritables inférences). Pendant
`rejouer`, le catalogue `forme → décision` est *consulté* (une table), pas le
cerveau — d'où 0. C'est tout l'amortissement.
</details>

<details><summary>Réponse 3</summary>
Un cerveau qui découpe toujours sans produire de traitement final fait croître la
file indéfiniment : sans borne, l'agent **tourne à l'infini** (temps et coût sans
fin). La safety bound arrête la découverte et signale l'échec.
</details>

## Pour aller plus loin

- [`Compression-07b`](../../Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) —
  l'organe dont ce carnet réutilise les primitives (le triage LLM complet, le
  rejeu sans LLM, la réfutation « le triage bat-il l'indifférencié ? »).
- L'EPIC #26930 — la suite : extraire ces primitives en **module importable**
  (le pas suivant nommé), puis composer un deuxième système (perception → action
  sur un autre domaine) pour montrer que le squelette est vraiment réutilisable.
